# Proyek Klasifikasi Gambar: Image Submission Classification
- **Nama:** Rois Hoiron
- **Email:** rois.khoiron@gmail.com
- **ID Dicoding:** khoironrois

## Import Semua Packages/Library yang Digunakan

In [5]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from pathlib import Path
import os
import kagglehub
AUTOTUNE = tf.data.AUTOTUNE

## Data Preparation

### Data Loading

In [6]:
# Download dataset from Kaggle
data_path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

# Load metadata and organize images by category
metadata_path = Path(data_path) / 'myntradataset' / 'styles.csv'
df = pd.read_csv(metadata_path, on_bad_lines="warn", engine="python")

# Use masterCategory as label (7 classes)
df['label'] = df['masterCategory'].astype('category')
df['label_id'] = df['label'].cat.codes
label_names = df["label"].cat.categories.tolist()
print(f'Classes: {label_names}')

# Create class directories and copy images
import shutil
from pathlib import Path
images_dir = Path(data_path) / "images"
class_base = Path(data_path) / "images_by_category"
if class_base.exists():
    shutil.rmtree(class_base)
class_base.mkdir()

for _, row in df.iterrows():
    img_file = images_dir / f"{row["id"]}.jpg"
    if img_file.exists():
        cat_dir = class_base / str(row["label_id"])
        cat_dir.mkdir(exist_ok=True)
        shutil.copy2(img_file, cat_dir / img_file.name)

data_dir = class_base
batch_size = 32
img_height, img_width = 180, 180

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
print(f'Found classes: {class_names}')


/var/folders/98/ls8tp8yx4c79w6nqhktq3vnw0000gn/T/ipykernel_95314/3030385429.py:6: ParserWarning: Skipping line 6044: Expected 10 fields in line 6044, saw 11

  df = pd.read_csv(metadata_path, on_bad_lines="warn", engine="python")
/var/folders/98/ls8tp8yx4c79w6nqhktq3vnw0000gn/T/ipykernel_95314/3030385429.py:6: ParserWarning: Skipping line 6569: Expected 10 fields in line 6569, saw 11

  df = pd.read_csv(metadata_path, on_bad_lines="warn", engine="python")
/var/folders/98/ls8tp8yx4c79w6nqhktq3vnw0000gn/T/ipykernel_95314/3030385429.py:6: ParserWarning: Skipping line 7399: Expected 10 fields in line 7399, saw 11

  df = pd.read_csv(metadata_path, on_bad_lines="warn", engine="python")
/var/folders/98/ls8tp8yx4c79w6nqhktq3vnw0000gn/T/ipykernel_95314/3030385429.py:6: ParserWarning: Skipping line 7939: Expected 10 fields in line 7939, saw 11

  df = pd.read_csv(metadata_path, on_bad_lines="warn", engine="python")
/var/folders/98/ls8tp8yx4c79w6nqhktq3vnw0000gn/T/ipykernel_95314/3030385429.py:6

Classes: ['Accessories', 'Apparel', 'Footwear', 'Free Items', 'Home', 'Personal Care', 'Sporting Goods']
Found 44419 files belonging to 7 classes.
Using 35536 files for training.
Found 44419 files belonging to 7 classes.
Using 8883 files for validation.
Found classes: ['0', '1', '2', '3', '4', '5', '6']


### Data Preprocessing

#### Split Dataset

In [7]:

def preprocess(image, label):
    return tf.cast(image, tf.float32) / 255.0, tf.cast(label, tf.int64)

train_ds = train_ds.map(preprocess).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.map(preprocess).cache().prefetch(AUTOTUNE)

# Split validation into val + test
test_ds = val_ds.take(200)
val_ds = val_ds.skip(200)


## Modelling

In [8]:
data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])
model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(img_height, img_width, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax'),
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

/opt/miniconda3/envs/codingskuy/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Evaluasi dan Visualisasi

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = callbacks.ModelCheckpoint('best_model.h5', save_best_only=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[early_stop, checkpoint])
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()

Epoch 1/50


I0000 00:00:1780293719.174390 11837401 shuffle_dataset_op.cc:453] ShuffleDatasetV3:16: Filling up shuffle buffer (this may take a while): 851 of 1000
I0000 00:00:1780293724.629909 11837401 shuffle_dataset_op.cc:483] Shuffle buffer filled.


## Konversi Model

In [ ]:
model.save('saved_model/')
converter = tf.lite.TFLiteConverter.from_saved_model('saved_model/')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
tfjs_converter_cmd = 'tensorflowjs_converter --input_format=tf_saved_model saved_model/ tfjs_model/'

## Inference (Optional)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test Accuracy: {test_acc:.2%}')